In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
from diabetes_inputs import FEATURES
from imblearn.over_sampling import RandomOverSampler
from imblearn.pipeline import Pipeline
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC

plt.style.use('fivethirtyeight')


In [ ]:
DATA_PATH = Path.cwd() / "diabetes.csv"
if not DATA_PATH.is_file():
    raise FileNotFoundError(
        f"Training data is not included. Place the authorized dataset at {DATA_PATH} "
        "or update DATA_PATH to its local location."
    )

df = pd.read_csv(DATA_PATH)
if df.columns.duplicated().any():
    raise ValueError("The dataset contains duplicate column names.")
required_columns = {*FEATURES, "Outcome"}
missing_columns = sorted(required_columns.difference(df.columns))
if missing_columns:
    raise ValueError(f"The dataset is missing required columns: {missing_columns}")

df = df.loc[:, [*FEATURES, "Outcome"]].copy()
for column in [*FEATURES, "Outcome"]:
    df[column] = pd.to_numeric(df[column], errors="coerce")
if df.empty or df.isna().any().any() or not np.isfinite(df.to_numpy(dtype=float)).all():
    raise ValueError("The dataset must contain finite numeric values and no missing data.")
if set(df["Outcome"].unique()) != {0, 1}:
    raise ValueError("Outcome must contain both binary labels 0 and 1.")


In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

In [ ]:
df.duplicated().sum()

In [ ]:
df.columns

In [ ]:
plt.figure(figsize=(20,10))
sns.heatmap(df.corr(),annot=True,fmt='0.1f',linewidth=0.7)
plt.show()

In [ ]:
sns.countplot(x='Outcome',data=df)
plt.show()

In [ ]:
sns.boxplot(x=df['Age'],data=df)
plt.show()

In [ ]:
X = df.loc[:, FEATURES]
y = df["Outcome"].astype("int64")


Install the pinned dependencies from `requirements.txt` before running the notebook. The training dataset (`diabetes.csv`) is not included. Sampling and model fitting below happen only after stratified train/validation/test splitting; validation is used for candidate selection, and the held-out test set is evaluated once at the end. The generated artifact has a distinct filename and does not replace the checked-in app model.


In [ ]:
# Reserve the test set before model comparison; use validation data for model choice only.
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.20, random_state=0, stratify=y
)
X_train, X_validation, y_train, y_validation = train_test_split(
    X_train_val, y_train_val, test_size=0.25, random_state=0, stratify=y_train_val
)

candidates = {
    "SVC": SVC(),
    "RandomForest": RandomForestClassifier(n_estimators=1000, random_state=0),
    "GradientBoosting": GradientBoostingClassifier(n_estimators=1000, random_state=0),
}
validation_f1 = {}
for name, estimator in candidates.items():
    candidate_pipeline = Pipeline([
        ("sampler", RandomOverSampler(random_state=0)),
        ("classifier", estimator),
    ])
    candidate_pipeline.fit(X_train, y_train)
    validation_f1[name] = f1_score(
        y_validation, candidate_pipeline.predict(X_validation), zero_division=0
    )
    print(f"{name} validation F1: {validation_f1[name]:.4f}")

best_name = max(validation_f1, key=validation_f1.get)
print(f"Selected using validation F1: {best_name}")

# Refit on train + validation only; the sampler is still inside the training pipeline.
final_model = Pipeline([
    ("sampler", RandomOverSampler(random_state=0)),
    ("classifier", clone(candidates[best_name])),
])
X_train_validation = pd.concat([X_train, X_validation])
y_train_validation = pd.concat([y_train, y_validation])
final_model.fit(X_train_validation, y_train_validation)

# Use the untouched test set once for a final local estimate, not external validation.
test_predictions = final_model.predict(X_test)
print("One-time held-out test estimate (not external or clinical validation)")
print(f"Accuracy: {accuracy_score(y_test, test_predictions):.4f}")
print(f"F1: {f1_score(y_test, test_predictions, zero_division=0):.4f}")

# Keep generated output separate from the trusted checked-in model.pkl.
MODEL_OUTPUT_PATH = Path.cwd() / "model_trained.pkl"
with MODEL_OUTPUT_PATH.open("wb") as model_file:
    import pickle
    pickle.dump(final_model, model_file)
print(f"Saved locally trained artifact to {MODEL_OUTPUT_PATH}")
